In [1]:
import pandas as pd
import numpy as np
import spacy

from natasha import (
    Segmenter,
    MorphVocab,
    
    NewsEmbedding,
    NewsMorphTagger,
    NewsSyntaxParser,
    NewsNERTagger,
    
    PER,
    NamesExtractor,

    Doc
)

import re

In [2]:
dataset = pd.read_csv("ru_cefr_short.csv")

In [3]:
dataset

,fragment,textbook-assigned cefr level
0,"Весной, летом и осенью почти каждую субботу он...",1
1,"Все говорят, что мама хорошая хозяйка. А ещё н...",1
2,На каждой двери красные плакаты и красные фона...,1
3,"Я считаю деньги, в час обедаю в кафе, а потом ...",1
4,Магазин «Чёрный квадрат» открывается в 9 часов...,1
...,...,...
7317,Утечка мозгов стала ключевым трендом междунаро...,6
7318,"По оценкам менеджеров «Промы», такая ситуация ...",6
7319,"Но это не мы, а техно-мемы заполоняют мир благ...",6
7320,Mapillary использует программное обеспечение д...,6


In [751]:
dataset.loc[dataset.duplicated(), :]

,fragment,textbook-assigned cefr level
797,Она живёт в маленькой квартире вместе с матерь...,2
805,"Музыкой начала заниматься, когда мне было 5 ле...",2
814,"Музыкой начала заниматься, когда мне было 5 ле...",2
817,До мая 2007 года об этой музыкальной группе ни...,2
818,До мая 2007 года об этой музыкальной группе ни...,2
...,...,...
6585,В 1952 году Шагал женится на Ваве — Валентине ...,5
6628,В мировой эпидемии ожирения виноват не только ...,5
6731,"И честнее было бы, чтобы герой ролика плыл по ...",5
6876,"Теперь же его знает каждый ребёнок»,— говорит ...",5


In [1020]:
dataset = dataset.loc[~dataset.duplicated(), :]
dataset

,fragment,textbook-assigned cefr level
0,"Весной, летом и осенью почти каждую субботу он...",1
1,"Все говорят, что мама хорошая хозяйка. А ещё н...",1
2,На каждой двери красные плакаты и красные фона...,1
3,"Я считаю деньги, в час обедаю в кафе, а потом ...",1
4,Магазин «Чёрный квадрат» открывается в 9 часов...,1
...,...,...
7317,Утечка мозгов стала ключевым трендом междунаро...,6
7318,"По оценкам менеджеров «Промы», такая ситуация ...",6
7319,"Но это не мы, а техно-мемы заполоняют мир благ...",6
7320,Mapillary использует программное обеспечение д...,6


In [1437]:
dataset["fragment"][7319]

'Но это не мы, а техно-мемы заполоняют мир благодаря машинерии, которая копирует, рекомбинирует, хранит и распространяет их. Это они стремительно эволюционируют, в то время как человеческие тела остаются прежними».'

In [4]:
dataset_2 = pd.read_csv("aggregated_results_by_ds__pool_37806285__2023_02_18.tsv", sep = "\t")
dataset_2 = dataset_2.drop("Unnamed: 3", axis = 1)
dataset_2

,INPUT:text,OUTPUT:category,CONFIDENCE:category
0,"Разглядывая их под микроскопом , мы увидим при...",[4],100.00%
1,"Иногда талантливые люди , не найдя себя в наук...",[4],100.00%
2,Кандидаты в депутаты от общественных организац...,[6],100.00%
3,"Главное , чтобы "" социальные лифты "" не остана...",[6],100.00%
4,Франция признает российских студентов и не при...,[2],100.00%
...,...,...,...
595,Характеристики солитонов при этом постоянно ме...,[5],100.00%
596,"Телефонами мы пользуемся помногу , в течение д...",[3],100.00%
597,Конкурентная оплата для преподавателя вуза в М...,[6],100.00%
598,"Главное , чтобы пункт назначения всегда был в ...",[5],100.00%


In [5]:
segmenter = Segmenter()
morph_vocab = MorphVocab()

emb = NewsEmbedding()
morph_tagger = NewsMorphTagger(emb)
syntax_parser = NewsSyntaxParser(emb)
ner_tagger = NewsNERTagger(emb)
names_extractor = NamesExtractor(morph_vocab)

In [6]:
dataset_np = np.array(dataset["fragment"])

In [841]:
dataset_np[6078]

'Но по статистике только из 7% собранного ПЭТ-пластика потом сделают новые бутылки. Для сравнения: 58% собранной бумаги и 70–90% металла превращаются в новые товары и продолжают служить людям.'

In [843]:
text = dataset_np[6078]
doc = Doc(text)

### Segmentation

In [846]:
doc.segment(segmenter)

In [848]:
doc.tokens

[DocToken(stop=2, text='Но'),
 DocToken(start=3, stop=5, text='по'),
 DocToken(start=6, stop=16, text='статистике'),
 DocToken(start=17, stop=23, text='только'),
 DocToken(start=24, stop=26, text='из'),
 DocToken(start=27, stop=28, text='7'),
 DocToken(start=28, stop=29, text='%'),
 DocToken(start=30, stop=40, text='собранного'),
 DocToken(start=41, stop=53, text='ПЭТ-пластика'),
 DocToken(start=54, stop=59, text='потом'),
 DocToken(start=60, stop=67, text='сделают'),
 DocToken(start=68, stop=73, text='новые'),
 DocToken(start=74, stop=81, text='бутылки'),
 DocToken(start=81, stop=82, text='.'),
 DocToken(start=83, stop=86, text='Для'),
 DocToken(start=87, stop=96, text='сравнения'),
 DocToken(start=96, stop=97, text=':'),
 DocToken(start=98, stop=100, text='58'),
 DocToken(start=100, stop=101, text='%'),
 DocToken(start=102, stop=111, text='собранной'),
 DocToken(start=112, stop=118, text='бумаги'),
 DocToken(start=119, stop=120, text='и'),
 DocToken(start=121, stop=126, text='70–90')

In [850]:
doc.sents

[DocSent(stop=82, text='Но по статистике только из 7% собранного ПЭТ-плас..., tokens=[...]),
 DocSent(start=83, stop=191, text='Для сравнения: 58% собранной бумаги и 70–90% мета..., tokens=[...])]

### Morphology

In [853]:
doc.tag_morph(morph_tagger)

In [855]:
doc.tokens

[DocToken(stop=2, text='Но', pos='CCONJ'),
 DocToken(start=3, stop=5, text='по', pos='ADP'),
 DocToken(start=6, stop=16, text='статистике', pos='NOUN', feats=<Inan,Dat,Fem,Sing>),
 DocToken(start=17, stop=23, text='только', pos='PART'),
 DocToken(start=24, stop=26, text='из', pos='ADP'),
 DocToken(start=27, stop=28, text='7', pos='NUM'),
 DocToken(start=28, stop=29, text='%', pos='SYM'),
 DocToken(start=30, stop=40, text='собранного', pos='VERB', feats=<Perf,Gen,Masc,Sing,Past,Part,Pass>),
 DocToken(start=41, stop=53, text='ПЭТ-пластика', pos='NOUN', feats=<Inan,Ins,Masc,Sing>),
 DocToken(start=54, stop=59, text='потом', pos='ADV', feats=<Pos>),
 DocToken(start=60, stop=67, text='сделают', pos='VERB', feats=<Perf,Ind,Plur,3,Fut,Fin,Act>),
 DocToken(start=68, stop=73, text='новые', pos='ADJ', feats=<Inan,Acc,Pos,Plur>),
 DocToken(start=74, stop=81, text='бутылки', pos='NOUN', feats=<Inan,Acc,Fem,Plur>),
 DocToken(start=81, stop=82, text='.', pos='PUNCT'),
 DocToken(start=83, stop=86, te

In [857]:
doc.sents[0].morph.print()

                  Но CCONJ
                  по ADP
          статистике NOUN|Animacy=Inan|Case=Dat|Gender=Fem|Number=Sing
              только PART
                  из ADP
                   7 NUM
                   % SYM
          собранного VERB|Aspect=Perf|Case=Gen|Gender=Masc|Number=Sing|Tense=Past|VerbForm=Part|Voice=Pass
        ПЭТ-пластика NOUN|Animacy=Inan|Case=Ins|Gender=Masc|Number=Sing
               потом ADV|Degree=Pos
             сделают VERB|Aspect=Perf|Mood=Ind|Number=Plur|Person=3|Tense=Fut|VerbForm=Fin|Voice=Act
               новые ADJ|Animacy=Inan|Case=Acc|Degree=Pos|Number=Plur
             бутылки NOUN|Animacy=Inan|Case=Acc|Gender=Fem|Number=Plur
                   . PUNCT


### Lemmatization

In [860]:
for token in doc.tokens:
     token.lemmatize(morph_vocab)

In [862]:
print(doc.tokens[:5])
{_.text: _.lemma for _ in doc.tokens}

[DocToken(stop=2, text='Но', pos='CCONJ', lemma='но'), DocToken(start=3, stop=5, text='по', pos='ADP', lemma='по'), DocToken(start=6, stop=16, text='статистике', pos='NOUN', feats=<Inan,Dat,Fem,Sing>, lemma='статистика'), DocToken(start=17, stop=23, text='только', pos='PART', lemma='только'), DocToken(start=24, stop=26, text='из', pos='ADP', lemma='из')]


{'Но': 'но',
 'по': 'по',
 'статистике': 'статистика',
 'только': 'только',
 'из': 'из',
 '7': '7',
 '%': '%',
 'собранного': 'собрать',
 'ПЭТ-пластика': 'пэт-пластик',
 'потом': 'потом',
 'сделают': 'сделать',
 'новые': 'новый',
 'бутылки': 'бутылка',
 '.': '.',
 'Для': 'для',
 'сравнения': 'сравнение',
 ':': ':',
 '58': '58',
 'собранной': 'собрать',
 'бумаги': 'бумага',
 'и': 'и',
 '70–90': '70–90',
 'металла': 'металл',
 'превращаются': 'превращаться',
 'в': 'в',
 'товары': 'товар',
 'продолжают': 'продолжать',
 'служить': 'служить',
 'людям': 'человек'}

### Syntax parser

In [865]:
doc.parse_syntax(syntax_parser)
print(doc.tokens[:5])
doc.sents[0].syntax.print()

[DocToken(stop=2, text='Но', id='1_1', head_id='1_11', rel='cc', pos='CCONJ', lemma='но'), DocToken(start=3, stop=5, text='по', id='1_2', head_id='1_3', rel='case', pos='ADP', lemma='по'), DocToken(start=6, stop=16, text='статистике', id='1_3', head_id='1_11', rel='obl', pos='NOUN', feats=<Inan,Dat,Fem,Sing>, lemma='статистика'), DocToken(start=17, stop=23, text='только', id='1_4', head_id='1_7', rel='advmod', pos='PART', lemma='только'), DocToken(start=24, stop=26, text='из', id='1_5', head_id='1_7', rel='case', pos='ADP', lemma='из')]
┌──────────► Но           cc
│         ┌► по           case
│ ┌──────►└─ статистике   obl
│ │   ┌────► только       advmod
│ │   │ ┌──► из           case
│ │   │ │ ┌► 7            nummod
│ │ ┌►└─└─└─ %            obl
│ │ │ │   ┌► собранного   amod
│ │ │ └──►└─ ПЭТ-пластика nmod
│ │ │     ┌► потом        advmod
└─└─└─┌─┌─└─ сделают      
      │ │ ┌► новые        amod
      │ └►└─ бутылки      obj
      └────► .            punct


### NER

In [868]:
doc.tag_ner(ner_tagger)
print(doc.spans[:5])
doc.ner.print()

[]
Но по статистике только из 7% собранного ПЭТ-пластика потом сделают 
новые бутылки. Для сравнения: 58% собранной бумаги и 70–90% металла 
превращаются в новые товары и продолжают служить людям.


## Features

In [1022]:
feats_table = pd.DataFrame()

In [1024]:
feats_table["length"] = dataset["fragment"].map(lambda x: len(x.split()))

In [1026]:
print(f'Среднее количество токенов в тексте: {feats_table["length"].mean()}')
print(f'Минимальное количество токенов в тексте: {feats_table["length"].min()}')
print(f'Максимальное количество токенов втексте: {feats_table["length"].max()}')

Среднее количество токенов в тексте: 24.061888352860098
Минимальное количество токенов в тексте: 15
Максимальное количество токенов втексте: 40


In [1030]:
feats_table["text"] = dataset["fragment"]
feats_table["complexity"] = dataset["textbook-assigned cefr level"]

In [879]:
sentence_test = doc.sents[0].text
sentence_test

'Но по статистике только из 7% собранного ПЭТ-пластика потом сделают новые бутылки.'

In [999]:
compute_sentence_info(doc.sents)

{1: {'df_tokens': {'1_1': ['Но', 'но', '1_11', 'cc', 'CCONJ', 2],
   '1_2': ['по', 'по', '1_3', 'case', 'ADP', 2],
   '1_3': ['статистике', 'статистика', '1_11', 'obl', 'NOUN', 10],
   '1_4': ['только', 'только', '1_7', 'advmod', 'PART', 6],
   '1_5': ['из', 'из', '1_7', 'case', 'ADP', 2],
   '1_6': ['7', '7', '1_7', 'nummod', 'NUM', 1],
   '1_7': ['%', '%', '1_11', 'obl', 'SYM', 1],
   '1_8': ['собранного', 'собрать', '1_9', 'amod', 'VERB', 10],
   '1_9': ['ПЭТ-пластика', 'пэт-пластик', '1_7', 'nmod', 'NOUN', 12],
   '1_10': ['потом', 'потом', '1_11', 'advmod', 'ADV', 5],
   '1_11': ['сделают', 'сделать', '1_0', 'root', 'VERB', 7],
   '1_12': ['новые', 'новый', '1_13', 'amod', 'ADJ', 5],
   '1_13': ['бутылки', 'бутылка', '1_11', 'obj', 'NOUN', 7],
   '1_14': ['.', '.', '1_11', 'punct', 'PUNCT', 1]},
  'df_heads': {'1_11': 6, '1_3': 1, '1_7': 4, '1_9': 1, '1_13': 1},
  'df_real_tokens': {'1_1': ['Но', 'но', '1_11', 'cc', 'CCONJ', 2],
   '1_2': ['по', 'по', '1_3', 'case', 'ADP', 2],
   

In [29]:
doc.sents

[DocSent(stop=82, text='Но по статистике только из 7% собранного ПЭТ-плас..., tokens=[...]),
 DocSent(start=83, stop=191, text='Для сравнения: 58% собранной бумаги и 70–90% мета..., tokens=[...])]

In [833]:
len(doc.sents[0].tokens[2].text)

5

In [158]:
from collections import Counter

In [1090]:
def compute_sentence_info(sentences):
    doc_sent_info = {}
    sent_n = 1
    for sentence in sentences:
        df_tokens = {}
        df_heads = {}
        df_lengths = {}
        for token in sentence.tokens:
            df_tokens[token.id] = [token.text, token.lemma, token.head_id, token.rel, token.pos, len(token.text)]
            if token.head_id not in df_heads:
                df_heads[token.head_id] = 1
            else:
                df_heads[token.head_id] += 1

        df_heads = {x:y for x,y in df_heads.items() if not x.endswith('_0')}
        doc_sent_info[sent_n] = {"df_tokens" : df_tokens, "df_heads": df_heads}
        df_real_tokens = {key : token for key, token in doc_sent_info[sent_n]["df_tokens"].items() if not token[4] == ('PUNCT' or "SYM" or "X")} # без знаков пунктуации
        doc_sent_info[sent_n]["df_real_tokens"] = df_real_tokens
        
        for token in sentence.tokens:
            head_id = doc_sent_info[sent_n]["df_tokens"][token.id][2]
            len_sent = 1
            tokens_done = []
            while not head_id.endswith("_0"):
                new_head_id = doc_sent_info[sent_n]["df_tokens"][head_id][2]
                if new_head_id in tokens_done:
                    doc_sent_info[sent_n]["df_heads"][new_head_id] -= 1 
                    new_head_id = str(sent_n) + "_0"
                    doc_sent_info[sent_n]["df_tokens"][head_id][2] = new_head_id
                    break
                tokens_done.append(head_id)
                head_id = new_head_id
                len_sent += 1
            df_lengths[token.id] = len_sent
        doc_sent_info[sent_n]["df_lengths"] = df_lengths

        double_child_n = Counter(doc_sent_info[sent_n]["df_heads"].values())[2]
        inner_vertices_n = len(doc_sent_info[sent_n]["df_heads"])
        vertices_n = len(doc_sent_info[sent_n]["df_real_tokens"])
        if len(doc_sent_info[sent_n]["df_heads"]) == 0:
            tree_depth = 0
        else:
            tree_depth = max(doc_sent_info[sent_n]["df_heads"].values())
        leaves_n = len(doc_sent_info[sent_n]["df_real_tokens"]) - len(doc_sent_info[sent_n]["df_heads"])
        
        dep_relations = np.array(list(doc_sent_info[sent_n]["df_real_tokens"].values()))[:,3]
        dep_relations = np.array([i.split(":")[0] for i in dep_relations])
        
        amod_n = Counter(dep_relations)['amod']
        compound_n = Counter(dep_relations)['compound']
        conj_n = Counter(dep_relations)['conj']
        det_n = Counter(dep_relations)['det']
        nmod_n = Counter(dep_relations)['nmod']
        nsubj_n = Counter(dep_relations)['nsubj']
    
        real_tokens_keys = np.array(list(doc_sent_info[sent_n]["df_real_tokens"].keys()))
        tokens_keys = np.array(list(doc_sent_info[sent_n]["df_tokens"].keys()))
        
        df_linear_distance = {}
        for token_id in doc_sent_info[sent_n]["df_real_tokens"]:
            token_head_id = doc_sent_info[sent_n]["df_real_tokens"][token_id][2]
            if token_head_id.endswith("_0"):
                lin_dist = 0
                pass
            else:
                if doc_sent_info[sent_n]["df_tokens"][token_head_id][4] == ('PUNCT' or "SYM" or "X"):
                    lin_dist = np.abs(np.where(real_tokens_keys == token_id)[0][0] - np.where(tokens_keys == token_head_id)[0][0])
                else:
                    lin_dist = np.abs(np.where(real_tokens_keys == token_id)[0][0] - np.where(real_tokens_keys == token_head_id)[0][0])
            df_linear_distance[token_id] = lin_dist
        
        mean_linear_distance = np.array(list(df_linear_distance.values())).mean()
        
        sw_array = np.array(list(df_real_tokens.values()))[:,5].astype(np.int_)
        asw = sw_array.mean()

        sentence_pos_array = np.array(list(df_real_tokens.values()))[:,4]
        
        adj_n = Counter(sentence_pos_array)['ADJ']
        adp_n = Counter(sentence_pos_array)['ADP']
        adv_n = Counter(sentence_pos_array)['ADV']
        aux_n = Counter(sentence_pos_array)['AUX']
        cconj_n = Counter(sentence_pos_array)['CCONJ']
        det_pos_n = Counter(sentence_pos_array)['DET']
        intj_n = Counter(sentence_pos_array)['INTJ']
        noun_n = Counter(sentence_pos_array)['NOUN']
        num_n = Counter(sentence_pos_array)['NUM']
        part_n = Counter(sentence_pos_array)['PART']
        pron_n = Counter(sentence_pos_array)['PRON']
        propn_n = Counter(sentence_pos_array)['PROPN']
        sconj_n = Counter(sentence_pos_array)['SCONJ']
        verb_n = Counter(sentence_pos_array)['VERB']
        
        doc_sent_info[sent_n]["double_child_n"] = double_child_n
        doc_sent_info[sent_n]["inner_vertices_n"] = inner_vertices_n
        doc_sent_info[sent_n]["vertices_n"] = vertices_n
        doc_sent_info[sent_n]["tree_depth"] = tree_depth
        doc_sent_info[sent_n]["leaves_n"] = leaves_n
        doc_sent_info[sent_n]["amod_n"] = amod_n
        doc_sent_info[sent_n]["compound_n"] = compound_n
        doc_sent_info[sent_n]["conj_n"] = conj_n
        doc_sent_info[sent_n]["det_n"] = det_n
        doc_sent_info[sent_n]["nmod_n"] = nmod_n
        doc_sent_info[sent_n]["nsubj_n"] = nsubj_n
        doc_sent_info[sent_n]["mean_linear_distance"] = mean_linear_distance
        
        doc_sent_info[sent_n]["asw"] = asw

        doc_sent_info[sent_n]["adj_n"] = adj_n / vertices_n
        doc_sent_info[sent_n]["adp_n"] = adp_n / vertices_n
        doc_sent_info[sent_n]["adv_n"] = adv_n / vertices_n
        doc_sent_info[sent_n]["aux_n"] = aux_n / vertices_n
        doc_sent_info[sent_n]["cconj_n"] = cconj_n / vertices_n
        doc_sent_info[sent_n]["det_pos_n"] = det_pos_n / vertices_n
        doc_sent_info[sent_n]["intj_n"] = intj_n / vertices_n
        doc_sent_info[sent_n]["noun_n"] = noun_n / vertices_n
        doc_sent_info[sent_n]["num_n"] = num_n / vertices_n
        doc_sent_info[sent_n]["part_n"] = part_n / vertices_n
        doc_sent_info[sent_n]["pron_n"] = pron_n / vertices_n
        doc_sent_info[sent_n]["propn_n"] = propn_n / vertices_n
        doc_sent_info[sent_n]["sconj_n"] = sconj_n / vertices_n
        doc_sent_info[sent_n]["verb_n"] = verb_n / vertices_n
        
        sent_n +=1
    return doc_sent_info

In [1373]:
def compute_sentence_features(sentence_info):
    double_child = []
    inner_vertices = []
    vertices = []
    tree_depth = []
    amod = []
    compound = []
    conj = []
    det = []
    nmod = []
    nsubj = []
    leaves = []
    mean_linear_distance = []

    adj = []
    adp = []
    adv = []
    aux = []
    cconj = []
    det_pos = []
    intj = []
    noun = []
    num = []
    part = []
    pron = []
    propn = []
    sconj = []
    verb = []
    
    for d in sentence_info.values():
        double_child.append(d["double_child_n"])
        inner_vertices.append(d["inner_vertices_n"])
        vertices.append(d["vertices_n"])
        tree_depth.append(d["tree_depth"])
        amod.append(d["amod_n"])
        compound.append(d["compound_n"])
        conj.append(d["conj_n"])
        det.append(d["det_n"])
        nmod.append(d["nmod_n"])
        nsubj.append(d["nsubj_n"])
        leaves.append(d["leaves_n"])
        mean_linear_distance.append(d["mean_linear_distance"])

        adj.append(d["adj_n"])
        adp.append(d["adp_n"])
        adv.append(d["adv_n"])
        aux.append(d["aux_n"])
        cconj.append(d["cconj_n"])
        det_pos.append(d["det_pos_n"])
        intj.append(d["intj_n"])
        noun.append(d["noun_n"])
        num.append(d["num_n"])
        part.append(d["part_n"])
        pron.append(d["pron_n"])
        propn.append(d["propn_n"])
        sconj.append(d["sconj_n"])
        verb.append(d["verb_n"])

    

    avg_double_child = np.mean(double_child)
    avg_inner_vertices = np.mean(inner_vertices)
    avg_vertices = np.mean(vertices)
    avg_tree_depth = np.mean(tree_depth)
    avg_amod = np.mean(amod)
    avg_compound = np.mean(compound)
    avg_conj = np.mean(conj)
    avg_det = np.mean(det)
    avg_nmod = np.mean(nmod)
    avg_nsubj = np.mean(nsubj)
    max_leaves = np.max(leaves)
    max_inner_vertices = np.max(inner_vertices)
    max_vertices = np.max(vertices)
    median_nmod = np.median(nmod)
    median_inner_vertices = np.median(inner_vertices)
    median_vertices = np.median(vertices)
    std_leaves = np.std(leaves)
    std_vertices = np.std(vertices)
    std_conj = np.std(conj)
    std_det = np.std(det)
    std_mean_linear_distance = np.std(mean_linear_distance)

    avg_adj = np.mean(adj)
    avg_adp = np.mean(adp)
    avg_adv = np.mean(adv)
    avg_aux = np.mean(aux)
    avg_cconj = np.mean(cconj)
    avg_det_pos = np.mean(det_pos)
    avg_intj = np.mean(intj)
    avg_noun = np.mean(noun)
    avg_num = np.mean(num)
    avg_part = np.mean(part)
    avg_pron = np.mean(pron)
    avg_propn = np.mean(propn)
    avg_sconj = np.mean(sconj)
    avg_verb = np.mean(verb)

    return {
        "avg_double_child": avg_double_child,
        "avg_inner_vertices": avg_inner_vertices,
        "avg_vertices": avg_vertices,
        "avg_tree_depth": avg_tree_depth,
        "avg_amod": avg_amod,
        "avg_compound": avg_compound,
        "avg_conj": avg_conj,
        "avg_det": avg_det,
        "avg_nmod": avg_nmod,
        "avg_nsubj": avg_nsubj,
        "max_leaves": max_leaves,
        "max_inner_vertices": max_inner_vertices,
        "max_vertices": max_vertices,
        "median_nmod": median_nmod,
        "median_inner_vertices": median_inner_vertices,
        "median_vertices": median_vertices,
        "std_leaves": std_leaves,
        "std_vertices": std_vertices,
        "std_conj": std_conj,
        "std_det": std_det,
        "std_mean_linear_distance": std_mean_linear_distance,
        "avg_adj": avg_adj,
        "avg_adp": avg_adp,
        "avg_adv": avg_adv,
        "avg_aux": avg_aux,
        "avg_cconj": avg_cconj,
        "avg_det_pos": avg_det_pos,
        "avg_intj": avg_intj,
        "avg_noun": avg_noun,
        "avg_num": avg_num,
        "avg_part": avg_part,
        "avg_pron": avg_pron,
        "avg_propn": avg_propn,
        "avg_sconj": avg_sconj,
        "avg_verb": avg_verb
        
    }

In [1348]:
def extract_features(text, segmenter=Segmenter(),
                     morph_vocab=MorphVocab(),
                     emb=NewsEmbedding(),
                     morph_tagger=NewsMorphTagger(emb),
                     syntax_parser=NewsSyntaxParser(emb),
                     ner_tagger=NewsNERTagger(emb), 
                     names_extractor=NamesExtractor(morph_vocab)):
    doc = Doc(text)
    doc.segment(segmenter)
    doc.tag_morph(morph_tagger)
    for token in doc.tokens:
        token.lemmatize(morph_vocab)
    doc.parse_syntax(syntax_parser)
    doc.tag_ner(ner_tagger)

    sentences = doc.sents
    sentence_info = compute_sentence_info(sentences)
    sentence_features = compute_sentence_features(sentence_info)
    features_df = sentence_features

    return features_df

In [1092]:
apply_df = feats_table.apply(lambda x: extract_features(x["text"]), axis=1, result_type="expand")

In [705]:
feats_table.loc[feats_table["text"] == "), стиль (одежда, её опрятность и уместность). И что важнее — большой вопрос. «Я считаю, что для карьерного роста главное — профессиональные навыки и знания."]

,length,text
5931,25,"), стиль (одежда, её опрятность и уместность)...."


In [1350]:
feats_table_final = pd.concat([feats_table, apply_df], axis=1)

In [1352]:
feats_table_final["length"] = feats_table_final["text"].map(lambda x: len(x.split()))

In [1354]:
ahaha = "NumPy is a Python library"
[m.start() for m in re.finditer('P', ahaha)]

[3, 11]

### Обучение

In [1357]:
from sklearn import preprocessing, svm
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, accuracy_score

In [1387]:
y = np.array(feats_table["complexity"]) - 1
X = np.array(feats_table_final.drop(columns=["text", "complexity", "length"]))
X_potest = X[:, :21]

In [1361]:
X

array([[ 0.25      ,  3.75      ,  7.        , ...,  0.        ,
         0.        ,  0.16260823],
       [ 0.5       ,  2.5       ,  6.75      , ...,  0.        ,
         0.04166667,  0.15833333],
       [ 1.        ,  3.        ,  7.        , ...,  0.        ,
         0.03125   ,  0.0625    ],
       ...,
       [ 1.        ,  5.5       , 14.5       , ...,  0.        ,
         0.04166667,  0.17156863],
       [ 1.5       ,  5.        , 10.        , ...,  0.03571429,
         0.03571429,  0.19047619],
       [ 2.5       ,  4.5       ,  9.        , ...,  0.1125    ,
         0.        ,  0.2125    ]])

In [1389]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state=30)
X_potest_train, X_potest_test, y_train, y_test = train_test_split(X_potest, y, test_size = 0.25, random_state=30)
regr = LinearRegression()
regr.fit(X_train, y_train)

y_pred_regr = regr.predict(X_test)
y_pred_regr_round = np.round(y_pred_regr)

In [1391]:
mse = mean_squared_error(y_test, y_pred_regr_round)
print("Linear Regression mse = ", mse)
print(accuracy_score(y_test, y_pred_regr_round))

Linear Regression mse =  1.326901874310915
0.3263506063947078


In [1369]:
columns = np.array(feats_table_final.drop(columns=["text", "complexity", "length"]).columns)

In [1457]:
feats_1 = feats_table_final.loc[round(feats_table_final["complexity"]) == 1].drop("text", axis=1)
feats_2 = feats_table_final.loc[round(feats_table_final["complexity"]) == 2].drop("text", axis=1)
feats_3 = feats_table_final.loc[round(feats_table_final["complexity"]) == 3].drop("text", axis=1)
feats_4 = feats_table_final.loc[round(feats_table_final["complexity"]) == 4].drop("text", axis=1)
feats_5 = feats_table_final.loc[round(feats_table_final["complexity"]) == 5].drop("text", axis=1)
feats_6 = feats_table_final.loc[round(feats_table_final["complexity"]) == 6].drop("text", axis=1)
all_feats = [feats_1.mean(), feats_2.mean(),  feats_3.mean(),  feats_4.mean(),  feats_5.mean(),  feats_6.mean()]
feats_df = pd.DataFrame(all_feats).transpose()
feats_df.columns = [1, 2, 3, 4, 5, 6]
feats_df

,1,2,3,4,5,6
length,24.946265,24.443643,24.083827,23.748535,23.470386,24.553333
complexity,1.000000,2.000000,3.000000,4.000000,5.000000,6.000000
avg_double_child,0.827560,1.181633,1.611756,1.843113,2.013877,2.133556
avg_inner_vertices,4.264104,5.338462,6.468452,7.030559,7.624492,7.699222
avg_vertices,9.209174,11.615344,13.415651,14.415475,15.820472,15.551444
avg_tree_depth,4.371312,4.671242,4.720562,4.795193,4.961688,5.118889
avg_amod,0.595368,0.834126,1.275649,1.603214,1.729757,1.386222
avg_compound,0.000000,0.000000,0.000635,0.001319,0.004292,0.000000
avg_conj,0.870111,0.863715,0.919771,1.087534,0.946381,1.049556
avg_det,0.214086,0.290017,0.382727,0.383724,0.509013,0.446889


### XGBoost

In [1139]:
import xgboost as xgb

In [1270]:
model = xgb.XGBClassifier(n_estimators = 100, random_state = 52)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_pred_round = np.round(y_pred)

mse = mean_squared_error(y_test, y_pred_round)
rmse = np.sqrt(mse)
print(mse)
print(rmse)
print(accuracy_score(y_test, y_pred_round))

1.659316427783903
1.2881445678897625
0.3533627342888644


In [1304]:
depth_list = [5, 7, 10]
max_leaf_list = [1, 2, 5, 10, 20, 100]
learning_rate_list = [0.01, 0.05, 0.1, 0.2, 0.4]
lambda_list = [1, 3, 5]

best_model_params = [5, 1, 0.8, 0.4, 1]
best_RMSE = 1.29
model_best = None

In [1306]:
import random

In [1308]:
random.seed(2025)
np.random.seed(2025)
etap = 0
for depth in depth_list:
    for lr in learning_rate_list:
        for lambda_ in lambda_list:
            for max_leaf in max_leaf_list:
                etap +=1
                model_new = xgb.XGBClassifier(n_estimators = 100, random_state = 52, max_depth=depth, max_leaves=max_leaf, learning_rate = lr, reg_lambda = lambda_)
                model_new.fit(X_train, y_train)
                preds_new = model_new.predict(X_test)
                RMSE_new = np.sqrt(np.mean((preds_new - y_test)**2))
                if RMSE_new < best_RMSE:
                    print(etap)
                    best_model_params = [depth, lr, lambda_, max_leaf]
                    best_RMSE = RMSE_new
                    model_best = model_new
                    print(f"So far best RMSE on validation: {best_RMSE}, params: depth = {depth}, lr = {lr}, lambda_ = {lambda_}, max_leaf = {max_leaf}")

1
So far best RMSE on validation: 1.2834284294916265, params: depth = 5, lr = 0.01, lambda_ = 1, max_leaf = 1
2
So far best RMSE on validation: 1.2758895550007798, params: depth = 5, lr = 0.01, lambda_ = 1, max_leaf = 2
3
So far best RMSE on validation: 1.25475940669436, params: depth = 5, lr = 0.01, lambda_ = 1, max_leaf = 5
4
So far best RMSE on validation: 1.2494761857270462, params: depth = 5, lr = 0.01, lambda_ = 1, max_leaf = 10
5
So far best RMSE on validation: 1.2294619254943817, params: depth = 5, lr = 0.01, lambda_ = 1, max_leaf = 20
23
So far best RMSE on validation: 1.2213643933146614, params: depth = 5, lr = 0.05, lambda_ = 1, max_leaf = 20
119
So far best RMSE on validation: 1.216615943926652, params: depth = 7, lr = 0.05, lambda_ = 3, max_leaf = 20


In [1286]:
best_model_params

[7, 0.05, 3, 20]

In [1296]:
best_depth = best_model_params[0]
best_lr = best_model_params[1]
best_lambda_ = best_model_params[2]
best_max_leaf = best_model_params[3]

In [1298]:
model_best = xgb.XGBClassifier(n_estimators = 100, 
                          random_state = 52, 
                          max_depth=best_depth, 
                          max_leaves=best_max_leaf, 
                          learning_rate = best_lr,
                          reg_lambda = best_lambda_)

In [1300]:
model_best.fit(X_train, y_train)
y_pred = model_best.predict(X_test)
y_pred_round = np.round(y_pred)

mse = mean_squared_error(y_test, y_pred_round)
rmse = np.sqrt(mse)
print(mse)
print(rmse)
print(accuracy_score(y_test, y_pred_round))

1.4801543550165381
1.216615943926652
0.38257993384785005
